Ryan Dunne C00263405

This project uses Googles pretrained Hand LandMarks CNN Model https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker
The User can show "Rock", "Paper" or "Scissors" to the camera in a game against a Bot supported by a Reinforced Learning Model.

The more you play against the bot, the more it learns your behaviour and the smarter it should get.

In [1]:
def get_fingers(hand_landmarks, h, w):

    landmarks = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

    fingers = {}
    
    # Thumb - compare x position (tip vs base)
    fingers['thumb'] = landmarks[4][0] < landmarks[3][0]  # flip for right hand
    
    # Other fingers - compare y position (tip vs middle joint)
    fingers['index']  = landmarks[8][1]  < landmarks[6][1]
    fingers['middle'] = landmarks[12][1] < landmarks[10][1]
    fingers['ring']   = landmarks[16][1] < landmarks[14][1]
    fingers['pinky']  = landmarks[20][1] < landmarks[18][1]
    
    return fingers

def find_gesture(fingers):

    if all(fingers.values()): # All 5 fingers extended
        return "Paper"

    if not any(fingers.values()): # No fingers extended
        return "Rock"

    if fingers['index'] and fingers['middle'] and not fingers['ring'] and not fingers['pinky'] and not fingers['thumb']: # Only index & middle
        return "Scissors"

In [2]:
import numpy as np
import random
import json
import os

choices = ["Rock", "Paper","Scissors"]
counters = {"Rock": "Paper", "Scissors": "Rock", "Paper": "Scissors"} # What counters each choice

class rlBot: # Q Learning Agent

    
    def __init__(self, lr= 0.1, gamma = 0.9, epsilon = 0.3):
        self.lr          = lr  
        self.gamma       = gamma
        self.epsilon     = epsilon
        self.q_table     = {}
        self.history     = []
        self.last_state  = None
        self.last_action = None

        if os.path.exists("rl_bot.json"):
            self.load()
        print("RL bot loaded from previous session.")

    def _get_state(self): # Uses last 3 player throws as state, padding Just needed for when there isn't any previous data
        padded = (["None"] * 3 + self.history)[-3:]
        return tuple(padded)

    def _get_q(self, state): # Creates a default entry of 0.0 for each choice if state not seen before
        if state not in self.q_table:
            self.q_table[state] = {c: 0.0 for c in choices}
        return self.q_table[state]

    def choose(self):
        state = self._get_state()
        self.last_state = state

        if random.random() < self.epsilon:
            action = random.choice(choices) # Explore Randomly

        else:
            q = self._get_q(state)
            action = max(q, key=q.get) # Exploit the best known action

        self.last_action = action
        return action

    def learn(self, player_choice, reward):
        if self.last_state is None:
            return

            
        self.history.append(player_choice)

        next_state = self._get_state()
        q_current  = self._get_q(self.last_state)[self.last_action]
        q_next_max = max(self._get_q(next_state).values())


        # Bellman equation - shift Q value towards new reward signal
        new_q = q_current + self.lr * (reward + self.gamma * q_next_max - q_current)
        self.q_table[self.last_state][self.last_action] = new_q

        # Reduce Epsilon so Bot explores less over time
        self.epsilon = max(0.05, self.epsilon * 0.99)



# Allows for persistence accross sessions
    def save(self):
        saveable = {str(k): v for k, v in self.q_table.items()}
        with open("rl_bot.json", "w") as f:
            json.dump(saveable, f)
        print("Bot progress saved.")

    def load(self):
        with open("rl_bot.json") as f:
            data = json.load(f)
        self.q_table = {eval(k): v for k, v in data.items()}

rlBot = rlBot()

RL bot loaded from previous session.


In [5]:
def print_summary(player_score, bot_score):
    from collections import Counter

    print("=" * 40)
    print("          GAME SUMMARY")
    print("=" * 40)

    result = "You won!" if player_score == 2 else "Bot won!"
    print(f"Result:   {result}  ({player_score} - {bot_score})")
    print(f"Rounds:   {len(rlBot.history)}")
    print(f"Epsilon:  {rlBot.epsilon:.3f} (lower = smarter)")

    print("\nYour throws:")
    counts = Counter(rlBot.history)
    for choice in ["Rock", "Paper", "Scissors"]:
        count = counts.get(choice, 0)
        pct   = (count / len(rlBot.history) * 100) if rlBot.history else 0
        print(f"  {choice:<10} {count} ({pct:.0f}%)")

    print("\nBot preferences:")
    for state, actions in rlBot.q_table.items():
        best = max(actions, key=actions.get)
        print(f"  After {list(state)} -> {best}")

    print("=" * 40)

In [11]:
import random
import time
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

latest_result = None

# -------Landmarker setup------------------
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)]


model_path = 'hand_landmarker.task'
BaseOptions          = mp.tasks.BaseOptions
HandLandmarker       = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
HandLandmarkerResult  = mp.tasks.vision.HandLandmarkerResult
VisionRunningMode    = mp.tasks.vision.RunningMode


def handle_result(result: HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    global latest_result
    latest_result = result




options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_hands=2,
    result_callback=handle_result
    )

cap = cv2.VideoCapture(0)

# ----------------------------------------------
def game():

    #Best of 3
    player_score = 0
    bot_score = 0
    roundNo = 1;
    choices = ["Rock", "Paper", "Scissors"]


    #Game States
    WAITING = "waiting"
    COUNTDOWN = "countdown"
    RESULT = "result"

    STATE = WAITING

    locked_gesture = None
    start_countdown = None
    result_text = ""
    result_display_start = None



    
    with HandLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened() and player_score < 2 and bot_score < 2:
            ret, frame = cap.read()
            if not ret:
                break

            timestamp_ms = int(time.time() * 1000)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            landmarker.detect_async(mp_image, timestamp_ms)

            gesture = "Unknown"
            h, w, _ = frame.shape

            if latest_result and latest_result.hand_landmarks:
                for hand_landmarks in latest_result.hand_landmarks:

                    # Draw connections
                    for start_idx, end_idx in HAND_CONNECTIONS:
                        start = hand_landmarks[start_idx]
                        end   = hand_landmarks[end_idx]
                        sx, sy = int(start.x * w), int(start.y * h)
                        ex, ey = int(end.x * w),   int(end.y * h)
                        cv2.line(frame, (sx, sy), (ex, ey), (0, 0, 255), 2)

                    # Draw dots
                    for landmark in hand_landmarks:
                        cx, cy = int(landmark.x * w), int(landmark.y * h)
                        cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

                    fingers = get_fingers(hand_landmarks, h, w)
                    gesture = find_gesture(fingers) or "Unknown"


            #UI for when in waiting state
            if STATE == WAITING:
                cv2.putText(frame, f"Round {roundNo}  |  You: {player_score}  Bot: {bot_score}",
                            (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
                cv2.putText(frame, f"Gesture: {gesture}",
                            (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)
                cv2.putText(frame, "Press SPACE to lock in",
                            (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)

                
                key = cv2.waitKey(1) & 0xFF
                if key == ord(' ') and gesture in choices:
                    locked_gesture = gesture
                    countdown_start = time.time()
                    STATE = COUNTDOWN
                elif key == ord('q'):
                    break

            #When in countdown state, counts down from three
            elif STATE == COUNTDOWN:
                elapsed   = time.time() - countdown_start
                remaining = int(3 - elapsed) + 1
                cv2.putText(frame, f"Locked: {locked_gesture}",
                            (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
                cv2.putText(frame, str(remaining),
                            (w//2 - 30, h//2), cv2.FONT_HERSHEY_SIMPLEX, 4, (0,0,255), 5)

                if elapsed >= 3:
                    bot_choice = rlBot.choose() # Bot chooses after countdown, determines outcome
                    
                    if locked_gesture == bot_choice:
                        reward = 0
                        result_text = f"Draw! Bot chose {bot_choice}"
                        
                    elif (locked_gesture == "Rock"     and bot_choice == "Scissors") or \
                         (locked_gesture == "Scissors" and bot_choice == "Paper")    or \
                         (locked_gesture == "Paper"    and bot_choice == "Rock"):
                        result_text = f"You win! Bot chose {bot_choice}"
                        player_score += 1
                        reward = -1
                        
                    else:
                        result_text = f"Bot wins! Bot chose {bot_choice}"
                        bot_score += 1
                        reward = 1

                    rlBot.learn(locked_gesture, reward)

                    roundNo += 1
                    result_display_start = time.time()
                    STATE = RESULT

                cv2.waitKey(1)


            
            #---------Result State-------
            
            elif STATE == RESULT:
                cv2.putText(frame, result_text,
                            (20, h//2), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)
                cv2.putText(frame, f"You: {player_score}  Bot: {bot_score}",
                            (20, h//2 + 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

                if time.time() - result_display_start >= 3:
                    STATE = WAITING

                cv2.waitKey(1)

            cv2.imshow("Rock Paper Scissors", frame)

        # ---- GAME OVER SCREEN ----
        winner = "You won the game!" if player_score == 2 else "Bot won the game!"
        
        print_summary(player_score, bot_score)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            cv2.putText(frame, winner,
                        (w//2 - 200, h//2), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255,0), 3)
            cv2.putText(frame, "Press Q to quit",
                        (w//2 - 100, h//2 + 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
            cv2.imshow("Rock Paper Scissors", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    rlBot.save()
    cap.release()
    cv2.destroyAllWindows()


game()

          GAME SUMMARY
Result:   Bot won!  (0 - 0)
Rounds:   28
Epsilon:  0.226 (lower = smarter)

Your throws:
  Rock       11 (39%)
  Paper      10 (36%)
  Scissors   7 (25%)

Bot preferences:
  After ['None', 'None', 'None'] -> Paper
  After ['None', 'None', 'Paper'] -> Rock
  After ['None', 'Paper', 'Rock'] -> Paper
  After ['Paper', 'Rock', 'Paper'] -> Paper
  After ['Rock', 'Paper', 'Paper'] -> Paper
  After ['Paper', 'Paper', 'Paper'] -> Scissors
  After ['Paper', 'Paper', 'Scissors'] -> Scissors
  After ['Paper', 'Scissors', 'Scissors'] -> Rock
  After ['Scissors', 'Scissors', 'Paper'] -> Paper
  After ['Scissors', 'Paper', 'Paper'] -> Rock
  After ['None', 'Paper', 'Scissors'] -> Rock
  After ['Paper', 'Scissors', 'Paper'] -> Paper
  After ['Scissors', 'Scissors', 'Scissors'] -> Rock
  After ['Scissors', 'Scissors', 'Rock'] -> Rock
  After ['Scissors', 'Rock', 'Paper'] -> Rock
  After ['Rock', 'Paper', 'Rock'] -> Rock
  After ['Paper', 'Rock', 'Scissors'] -> Paper
  After ['Ro

========================================
          GAME SUMMARY 1
========================================
Result:   You won!  (2 - 1)
Rounds:   10
Epsilon:  0.271 (lower = smarter)

Your throws:
  Rock       0 (0%)
  Paper      5 (50%)
  Scissors   5 (50%)

Bot preferences:
  After ['None', 'None', 'None'] -> Paper
  After ['None', 'None', 'Paper'] -> Rock
  After ['None', 'Paper', 'Rock'] -> Paper
  After ['Paper', 'Rock', 'Paper'] -> Paper
  After ['Rock', 'Paper', 'Paper'] -> Paper

========================================
          GAME SUMMARY 2
========================================
Result:   Bot won!  (1 - 2)
Rounds:   15
Epsilon:  0.258 (lower = smarter)

Your throws:
  Rock       2 (13%)
  Paper      6 (40%)
  Scissors   7 (47%)

Bot preferences:
  After ['None', 'None', 'None'] -> Paper
  After ['None', 'None', 'Paper'] -> Rock
  After ['None', 'Paper', 'Rock'] -> Paper
  After ['Paper', 'Rock', 'Paper'] -> Paper
  After ['Rock', 'Paper', 'Paper'] -> Paper

========================================
          GAME SUMMARY 3
========================================
Result:   You won!  (2 - 0)
Rounds:   17
Epsilon:  0.253 (lower = smarter)

Your throws:
  Rock       2 (12%)
  Paper      8 (47%)
  Scissors   7 (41%)

Bot preferences:
  After ['None', 'None', 'None'] -> Paper
  After ['None', 'None', 'Paper'] -> Rock
  After ['None', 'Paper', 'Rock'] -> Paper
  After ['Paper', 'Rock', 'Paper'] -> Paper
  After ['Rock', 'Paper', 'Paper'] -> Paper

========================================
          GAME SUMMARY 4
========================================
Result:   Bot won!  (0 - 2)
Rounds:   19
Epsilon:  0.248 (lower = smarter)

Your throws:
  Rock       2 (11%)
  Paper      10 (53%)
  Scissors   7 (37%)

Bot preferences:
  After ['None', 'None', 'None'] -> Paper 
  After ['Scissors', 'Rock', 'Paper'] -> Rock
  After ['Rock', 'Paper', 'Rock'] -> Rock
  After ['Paper', 'Rock', 'Scissors'] -> Paper
  After ['Rock', 'Scissors', 'Paper'] -> Paper

========================================
          GAME SUMMARY 5
========================================
Result:   You won!  (2 - 0)
Rounds:   28
Epsilon:  0.226 (lower = smarter)

Your throws:
  Rock       11 (39%)
  Paper      10 (36%)
  Scissors   7 (25%)

Bot preferences:
  After ['Paper', 'Rock', 'Scissors'] -> Paper
  After ['Rock', 'Scissors', 'Paper'] -> Paper
  After ['Paper', 'Paper', 'Rock'] -> Rock
  After ['Paper', 'Rock', 'Rock'] -> Rock
  After ['Rock', 'Rock', 'Rock'] -> Rock